# Batched vs Per-Agent: Step-by-Step Comparison

In [14]:
using Pkg
Pkg.activate("../")

  Activating project at `~/workspace/ExchangeMarket.jl/scripts`


In [15]:
using Revise
using Random, SparseArrays, LinearAlgebra, Printf
using ExchangeMarket
include("./setup.jl")

Dict{Symbol, Symbol} with 6 entries:
  :LogBarPdPCG => :rect
  :LogBarPCG   => :rect
  :LogBar      => :circle
  :Tât         => :rect
  :PropRes     => :rect
  :PathFol     => :dtriangle

In [16]:
Random.seed!(1)
n = 3
m = 5
ρ = 0.5

f0 = FisherMarket(m, n; ρ=ρ, bool_unit=true, scale=30.0, sparsity=0.2)
linconstr = LinearConstr(1, n, ones(1, n), [sum(f0.w)])
p₀ = ones(n) * sum(f0.w) / n

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0001 seconds


3-element Vector{Float64}:
 0.3333333333333333
 0.3333333333333333
 0.3333333333333333

## Create two copies: f_agent (per-agent) and f_batch (batched workspace)

In [17]:
(name, method, kwargs) = method_kwargs[1]  # LogBar: DRq + pred_corr + logbar

# Per-agent market
f_agent = copy(f0)
f_agent.x .= ones(n, m) ./ m
f_agent.p .= p₀
alg_agent = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)

# Batched market
f_batch = copy(f0)
f_batch.x .= ones(n, m) ./ m
f_batch.p .= p₀
f_batch.workspace = cpu_workspace(f_batch)
alg_batch = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)

println("f_agent.workspace = ", f_agent.workspace)
println("f_batch.workspace = ", typeof(f_batch.workspace))

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0001 seconds
FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0000 seconds
FisherMarket initialized in 0.0001 seconds
f_agent.workspace = nothing
f_batch.workspace = MarketWorkspace{Matrix{Float64}, Vector{Float64}}


## Compare play! only (before init!)

In [19]:
# Compare a single play! call at the same price
p_test = ones(n) * 0.5

# agent play!
f_a = copy(f0); f_a.x .= ones(n,m)./m
alg_a = method(n, m, copy(p_test); linconstr=linconstr, kwargs...)
play!(alg_a, f_a; all=true)

# batch play!
f_b = copy(f0); f_b.x .= ones(n,m)./m
f_b.workspace = cpu_workspace(f_b)
alg_b = method(n, m, copy(p_test); linconstr=linconstr, kwargs...)
play!(alg_b, f_b; all=true)

@printf("After single play!:\n")
@printf("  |x diff|    = %.2e\n", maximum(abs.(f_a.x .- f_b.x)))
@printf("  |val_u diff|= %.2e\n", maximum(abs.(f_a.val_u .- f_b.val_u)))
@printf("  |sumx diff| = %.2e\n", maximum(abs.(f_a.sumx .- f_b.sumx)))
@printf("  sampler batchsize: agent=%d  batch=%d\n", alg_a.sampler.batchsize, alg_b.sampler.batchsize)

println("\nAgent x[:,1] = ", f_a.x[:, 1])
println("Batch x[:,1] = ", f_b.x[:, 1])
println("\nAgent val_u = ", f_a.val_u)
println("Batch val_u = ", f_b.val_u)
println("\nAgent sumx = ", f_a.sumx)
println("Batch sumx = ", f_b.sumx)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0022 seconds
FisherMarket initialized in 0.0022 seconds
FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0001 seconds
After single play!:
  |x diff|    = 5.55e-17
  |val_u diff|= 0.00e+00
  |sumx diff| = 5.55e-17
  sampler batchsize: agent=5  batch=5

Agent x[:,1] = [0.09566141457484352, 0.0, 0.0]
Batch x[:,1] = [0.09566141457484352, 0.0, 0.0]

Agent val_u = [32.66278386750295, 15.690067491742584, 0.0, 0.03154802837767862, 61.69909674999112]
Batch val_u = [32.66278386750295, 15.690067491742584, 0.0, 0.03154802837767862, 61.69909674999112]

Agent sumx = [0.3213751492752642, 0.7042847821373642, 0.4550189061647012]
Batch sumx = [0.3213751492752642, 0.7042847821373642, 0.45501890616470114]


## init! — compare after initialization

In [20]:
ExchangeMarket.init!(alg_agent, f_agent)
ExchangeMarket.init!(alg_batch, f_batch)

@printf("After init!:\n")
@printf("  |p diff|    = %.2e\n", maximum(abs.(alg_agent.p .- alg_batch.p)))
@printf("  |x diff|    = %.2e\n", maximum(abs.(f_agent.x .- f_batch.x)))
@printf("  |∇ diff|    = %.2e\n", maximum(abs.(alg_agent.∇ .- alg_batch.∇)))
@printf("  |sumx diff| = %.2e\n", maximum(abs.(f_agent.sumx .- f_batch.sumx)))
@printf("  |val_u diff|= %.2e\n", maximum(abs.(f_agent.val_u .- f_batch.val_u)))
@printf("  μ: agent=%.4e  batch=%.4e\n", alg_agent.μ, alg_batch.μ)
@printf("  φ: agent=%.6e  batch=%.6e\n", alg_agent.φ, alg_batch.φ)

found initial μ: 102.4: comp/μ: 0.0043845950111259845
found initial μ: 102.4: comp/μ: 0.0043845950111259845
After init!:
  |p diff|    = 0.00e+00
  |x diff|    = 0.00e+00
  |∇ diff|    = 0.00e+00
  |sumx diff| = 0.00e+00
  |val_u diff|= 0.00e+00
  μ: agent=1.0240e+02  batch=1.0240e+02
  φ: agent=-Inf  batch=-Inf


## Iterate step-by-step

In [21]:
for k in 1:5
    ExchangeMarket.iterate!(alg_agent, f_agent)
    ExchangeMarket.iterate!(alg_batch, f_batch)
    
    @printf("\n--- iter %d ---\n", k)
    @printf("  |p diff|    = %.2e\n", maximum(abs.(alg_agent.p .- alg_batch.p)))
    @printf("  |x diff|    = %.2e\n", maximum(abs.(f_agent.x .- f_batch.x)))
    @printf("  |∇ diff|    = %.2e\n", maximum(abs.(alg_agent.∇ .- alg_batch.∇)))
    @printf("  |sumx diff| = %.2e\n", maximum(abs.(f_agent.sumx .- f_batch.sumx)))
    @printf("  |val_u diff|= %.2e\n", maximum(abs.(f_agent.val_u .- f_batch.val_u)))
    @printf("  gₙ:  agent=%.4e  batch=%.4e\n", alg_agent.gₙ, alg_batch.gₙ)
    @printf("  α:   agent=%.4e  batch=%.4e\n", alg_agent.α, alg_batch.α)
    @printf("  μ:   agent=%.4e  batch=%.4e\n", alg_agent.μ, alg_batch.μ)
    @printf("  φ:   agent=%.6e  batch=%.6e\n", alg_agent.φ, alg_batch.φ)
end


--- iter 1 ---
  |p diff|    = 0.00e+00
  |x diff|    = 0.00e+00
  |∇ diff|    = 0.00e+00
  |sumx diff| = 0.00e+00
  |val_u diff|= 0.00e+00
  gₙ:  agent=1.7279e+00  batch=1.7279e+00
  α:   agent=9.9950e-01  batch=9.9950e-01
  μ:   agent=3.8095e-01  batch=3.8095e-01
  φ:   agent=3.051145e+02  batch=3.051145e+02

--- iter 2 ---
  |p diff|    = 5.55e-17
  |x diff|    = 5.55e-17
  |∇ diff|    = 1.11e-16
  |sumx diff| = 5.55e-17
  |val_u diff|= 2.78e-17
  gₙ:  agent=7.1450e-01  batch=7.1450e-01
  α:   agent=9.6125e-01  batch=9.6125e-01
  μ:   agent=1.9440e-02  batch=1.9440e-02
  φ:   agent=3.201580e+00  batch=3.201580e+00

--- iter 3 ---
  |p diff|    = 5.55e-17
  |x diff|    = 2.22e-16
  |∇ diff|    = 2.22e-16
  |sumx diff| = 2.22e-16
  |val_u diff|= 1.07e-14
  gₙ:  agent=5.3337e-01  batch=5.3337e-01
  α:   agent=9.9950e-01  batch=9.9950e-01
  μ:   agent=2.7037e-04  batch=2.7037e-04
  φ:   agent=3.129057e+00  batch=3.129057e+00

--- iter 4 ---
  |p diff|    = 5.55e-17
  |x diff|    = 1.11

## Full opt! comparison

In [22]:
# Fresh runs with full opt!
f_a2 = copy(f0); f_a2.x .= ones(n,m)./m; f_a2.p .= p₀
alg_a2 = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
opt!(alg_a2, f_a2; keep_traj=true, maxiter=20)
@printf("\nPer-agent: gₙ=%.2e\n", alg_a2.gₙ)
validate(f_a2, alg_a2)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0001 seconds
--------------------------------------------------------------------------------------------
                   ExchangeMarket.jl: A Julia Package for Exchange Market                   
                                    © Chuwen Zhang (2024)                                    
--------------------------------------------------------------------------------------------
 subproblem solver alias       := CESAnalytic
 subproblem solver style       := analytic
 lin-system solver alias       := DRq
 option for step               := logbar
 option for μ                  := pred_corr
--------------------------------------------------------------------------------------------
running Phase I...
      k |  lg(μ) |             φ |    |∇φ| |    |Δp| |      Δu |       t |      tₗ |      α 
found initial μ: 102.4: comp/μ: 0.0043845950111259845
      0 |  +2.01 | 

In [24]:
f_b2 = copy(f0); f_b2.x .= ones(n,m)./m; f_b2.p .= p₀
f_b2.workspace = cpu_workspace(f_b2)
alg_b2 = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
opt!(alg_b2, f_b2; keep_traj=true, maxiter=20)
@printf("\nBatched: gₙ=%.2e\n", alg_b2.gₙ)
validate(f_b2, alg_b2)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0021 seconds
FisherMarket initialized in 0.0022 seconds
--------------------------------------------------------------------------------------------
                   ExchangeMarket.jl: A Julia Package for Exchange Market                   
                                    © Chuwen Zhang (2024)                                    
--------------------------------------------------------------------------------------------
 subproblem solver alias       := CESAnalytic
 subproblem solver style       := analytic
 lin-system solver alias       := DRq
 option for step               := logbar
 option for μ                  := pred_corr
--------------------------------------------------------------------------------------------
running Phase I...
      k |  lg(μ) |             φ |    |∇φ| |    |Δp| |      Δu |       t |      tₗ |      α 
found initial μ: 102.4: comp/μ: 0.0043845950111259845
      0 |  +2.01 | 